Q learning, main purpose is to learn value of action, then using action value to determine policy.

Policy is the ultimate goal, but not the **directly** learned object.

---
### REINFORCE Intro
> Why not make learning policy the main task?
>
> Can we use sampled rewards to update the policy in the direction of higher expected reward?

our goal:
$$
\max_\theta J(\theta)
$$

how to calculate gradient:

$$
\nabla_\theta J
=
\mathbb E\left[
R\,\nabla_\theta\log\pi_\theta(a|s)
\right]
$$

We  want **gradient ascend**, Pytorch only provides **gradient descend** so we use a **surrogate loss**:

The goal is to minizie:
> loss = -policy.log_prob(action) * reward

We know this will increase the action's probablity (covered in llm series).
> min(-policy.log_prob(action))

the larger the reward, should affect more:
> min(-policy.log_prob(action) * reward)

An example:
```text
dance -> +1
forward -> +10
```

If agent picks dance, it will increase its P (according to equation above). You say:
>but dance is worse!

Yes, if we pick forward, its P will increase more!

REINFORCE doesn't need **every** sample update to be correct. It says: when do it enough time, the expected sample updates is J's gradient.

# REINFORCE (Williams, 1992)

**Core idea:** Optimize a stochastic policy directly using sampled rewards, without first learning a value function.

## Theorem 1 — Immediate Reward

**Master these three levels first. Otherwise, Policy Gradient can get very confusing!**

### 1. Parameterized Policy Family

A Policy Family is a function that maps parameters $\theta$ to a policy:

$$
F:\theta\mapsto\pi_\theta
$$

- It takes $\theta$ as input and produces a **policy function**. In other words, it's a higher-order function: a function that returns another function!
- The Policy Family determines which policies can be represented.
- Think of it like an **LLM architecture**. Different architectures have different representational capabilities. Changing the architecture changes what the model can potentially learn. Training adjusts the parameters to find a better function within that architecture.
- Similarly, in RL, the Policy Family defines the space of possible policies, while training searches for better parameters within that space.

### 2. Current Policy

Given a particular $\theta$, we get one specific policy:

$$
\pi_\theta(a\mid s)
$$

- The Policy Family contains all policies representable by the parameterization.
- The current $\theta$ selects **one policy** from that family.
- As training updates $\theta$, the current policy changes, but the **Policy Family stays the same** (assuming we don't change the architecture).

### 3. Evaluation Function — $J$

Given a Policy Family and a fixed environment, we get a corresponding objective function $J$.

$$
J(\theta)=\sum_a\pi_\theta(a)R(a)
$$


Ex.

```python
import math

def sigmoid(x):
    return 1 / (1 + math.exp(-x))

def policy(theta):
    return sigmoid(theta)

def J(theta):
    p = policy(theta)
    return 10 * p + 2 * (1 - p)

for theta in [-1, 0, 1]:
    print(f"theta={theta:2d}, p={policy(theta):.3f}, J={J(theta):.3f}")
```

The question:
>If $J$ is only implicitly defined by the policy family and the environment, can we still optimize it using gradient ascent without knowing its explicit analytical form?
>
>**REINFORCE shows that we can.**



REINFORCE update:

$$
\Delta\theta=\alpha R\nabla_\theta\log\pi_\theta(a\mid s)
$$

**Guarantee:** Under the theorem's assumptions, the expected update follows the **true reward (a.k.a the expected reward) gradient**.

$$
\mathbb E[\Delta\theta_t\mid\theta_t=\theta]
=
\alpha\nabla_\theta J(\theta)
$$

But the equation also shows its limitation/issue:
>Unbiased in expectation doesn't mean reliable in any **single update**. And since the parameters **change after every update**, we never actually get to **observe that expectation**!
>
>The equation says **expected value**, which needs many samples, but we only sample once, then update!
>
>1 way to deal with noise is decreasing learning rate

a.k.a

$$
\begin{aligned}
&\text{REINFORCE: unbiased gradient}\\
&\quad\downarrow\\
&\text{Robbins--Monro: control accumulated noise}\\
&\quad\downarrow\\
&\text{Stability assumptions: convergence}
\end{aligned}
$$

also,

$$
\text{Unbiased gradient}
\;\not\Rightarrow\;
\text{Parameter convergence}
\;\not\Rightarrow\;
\text{Optimal policy}
$$

---

## Theorem 2 — Delayed Reward

**Problem:** What if multiple actions are taken before receiving a reward?

**Example:**

The robot must press two buttons in sequence. It receives a reward only after both choices.

- A → A: Reward = 1
- A → B: Reward = 0
- B → A: Reward = 0
- B → B: Reward = 0

The robot receives no feedback after the first action.

**How can the final reward be used to update the policy for both actions?**

Accumulate the log-probability gradients and multiply by the final reward:

$$
\boxed{
\Delta\theta=\alpha R\sum_{t=1}^{T}
\nabla_\theta\log\pi_\theta(a_t\mid s_t)
}
$$

**Why addition?**

- Trajectory probability = product of action probabilities (and environment terms).
- Log turns products into sums.
- Gradient of a sum = sum of gradients.

Under the theorem's episodic assumptions, the expected update still follows the correct gradient.

## Limitations

- **Unbiased ≠ guaranteed convergence.**
- Individual updates can be noisy; variance can be very high.
- A final reward does not tell us which actions were responsible.
- Baselines and value-function estimates can reduce variance, motivating Actor-Critic methods.

**Takeaway:** REINFORCE provides a mathematically justified way to learn a policy directly from sampled rewards, even when the environment is unknown or non-differentiable.

### Issue: high variance during gradient ascend

For example, say each time check gradient is like checking compass. Say you get results: [200, 100, -100, -200], the right direction is 0.

The updates **fluctuate significantly** even though their expectation is zero. High variance makes optimization noisy and can require more samples to make reliable progress.

Lots of back and forth, wasted effort. This will make the journey very inefficient.

How to solve this?

Baseline is a nature idea:

dance -> +1
forward -> +10

why we still need to increase dance?

We can have a baseline, if you are better -> increase, worse -> decrease P.

and subtracting an **action-independent** baseline **does not change** the expected policy gradient.:

$$
\mathbb E[(R-b)\nabla\log\pi]
=
\mathbb E[R\nabla\log\pi]
$$


<br><br>

#### What should be baseline?

Say a state has 3 actions, the baseline should be the **state's expected return** (average action's return in this state).

baseline need to be **action-independent**. Compute Advantage:
$$
A_t=G_t-V(s_t)
$$
using a baseline that has **NOT been updated** with the current sample.
